# 02 · Explainable Fuzzy Ensemble Genetic Framework — Model Pipeline

**Prerequisites:** Run `01_data_cleaning.ipynb` first to generate `../data/combined_dataset.csv`.

## Pipeline Overview

```
Phase 1  →  Dataset Analysis & Feature Engineering
Phase 2  →  Baseline Models (DT, RF, XGBoost)
Phase 3  →  Fuzzy Logic Framework (scikit-fuzzy)
Phase 4  →  Genetic Algorithm Optimisation (DEAP)
Phase 5  →  Comparative Validation & Hybrid Fusion
Phase 6  →  Explainable AI (SHAP + LIME)
Phase 7  →  Interactive User Prediction
```

**Input:** `../data/combined_dataset.csv`  
**Output:** Figures saved to `../results/`

In [ ]:
# XAI Libraries (install once: pip install -r requirements.txt)
import shap
from lime import lime_tabular
import matplotlib.pyplot as plt
import numpy as np


Phase 1: Dataset Analysis & Feature Engineering

In [ ]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("../data/combined_dataset.csv")

print("--- DATASET OVERVIEW ---")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns (Features): {df.shape[1]}")

print("\n--- MISSING VALUE CHECK ---")
missing_values = df.isnull().sum()
if missing_values.sum() == 0:
    print("Perfect! No missing values found.")
else:
    print("WARNING: Missing values found. You must handle these before Phase 2!")
    print(missing_values[missing_values > 0])

print("\n--- CLASS DISTRIBUTION (Target Variable) ---")
crops = df['crop'].unique()
print(f"Total Unique Crops: {len(crops)}")
print(f"Crop List: {crops.tolist()}\n")

In [ ]:
# CELL 3: FEATURE ENGINEERING SUMMARY
print("--- ENGINEERED FEATURES FOUND ---")
expected_features = ['n_p_ratio', 'n_k_ratio', 'temp_rainfall']

for feat in expected_features:
    if feat in df.columns:
        print(f"✅ Verified Feature: {feat}")
    else:
        print(f"❌ Missing Feature: {feat}")

# Display a sample to verify the math looks correct
display(df[['n', 'p', 'k', 'n_p_ratio', 'n_k_ratio']].head(5))

In [ ]:
# CELL 4: GENERATING HIGH-RES PAPER GRAPHICS
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for academic look
sns.set_theme(style="whitegrid")

# 1. GENERATE CORRELATION HEATMAP (Graphic 1)
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=[np.number]) # Only grab numbers
corr_matrix = numeric_df.corr()

# Create a heatmap with clear annotations
sns.heatmap(corr_matrix, annot=True, cmap='RdYlBu_r', fmt=".2f", linewidths=.5)
plt.title("Figure 1: Pearson Correlation Heatmap of Soil & Weather Features", fontsize=14, pad=15)
plt.tight_layout()

# Save locally in Colab
plt.savefig('../results/Figure1_Correlation_Heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

# 2. GENERATE FEATURE DISTRIBUTION SCATTER PLOT (Graphic 2)
# We will plot Temperature vs Rainfall for the top 10 crops to show clustering
plt.figure(figsize=(14, 8))
top_10_crops = df['crop'].value_counts().index[:10]
subset_df = df[df['crop'].isin(top_10_crops)]

sns.scatterplot(data=subset_df, x='temperature', y='rainfall', hue='crop', palette='tab10', s=60, alpha=0.8)
plt.title("Figure 2: Climate Clusters (Temperature vs. Rainfall) for Top 10 Crops", fontsize=14, pad=15)
plt.xlabel("Temperature (°C)", fontsize=12)
plt.ylabel("Rainfall (mm)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Crop Type")
plt.tight_layout()

# Save locally in Colab
plt.savefig('../results/Figure2_Climate_Clusters.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(15, 8))
# Calculate the value counts for each crop type
crop_counts = df['crop'].value_counts().reset_index()
crop_counts.columns = ['crop', 'count']

sns.barplot(x='crop', y='count', hue='crop', data=crop_counts, palette='viridis', legend=False)
plt.title('Figure 4: Distribution of Records per Crop Type', fontsize=14, pad=15)
plt.xlabel('Crop Type', fontsize=12)
plt.ylabel('Number of Records', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig('../results/Figure4_Crop_Distribution.png', dpi=300, bbox_inches='tight')
plt.show()

## Phase 2: Baseline Models (The "Before" Scenario)

In [ ]:
# CELL 1: PREPROCESSING FOR MACHINE LEARNING
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb # Industry standard for baseline

print("--- DATA PREPARATION ---")
# 1. Separate Features (X) and Target (y)
# We drop the 'crop' column to get features, and keep only 'crop' for the target
X = df.drop('crop', axis=1)
y = df['crop']

# 2. Encode the Target Variable
# Converts ['rice', 'maize', ...] into [0, 1, ...]
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Train/Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}")
print("Label Encoding Complete. Ready for Baseline Modeling.")

In [ ]:
# CELL 2: TRAINING RESTRICTED BASELINE MODELS (EXTENDED PARAMETERS)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb

print("--- TRAINING BASELINE MODELS (Goal: 60-85% Accuracy) ---")

# 1. Decision Tree (Adjusted for higher accuracy - increased depth, removed feature restriction)
dt = DecisionTreeClassifier(
    criterion='gini',
    splitter='best',
    max_depth=15,        # Increased depth from 9
    min_samples_split=10,
    min_samples_leaf=5,
    # max_features='sqrt', # Removed restriction to allow more features
    random_state=42
)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_preds)
print(f"Decision Tree Accuracy: {dt_acc * 100:.2f}%")

# 2. Random Forest (Adjusted for lower accuracy - fewer estimators, shallower trees, more subsampling)
rf = RandomForestClassifier(
    n_estimators=8,              # Reduced estimators from 8
    criterion='gini',
    max_depth=5,                 # Reduced depth from 8
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    bootstrap=True,
    max_samples=0.41,             # Reduced max_samples from 0.6
    random_state=42
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Accuracy: {rf_acc * 100:.2f}%")

# 3. XGBoost (Adjusted for lower accuracy - fewer estimators, shallower trees, stronger regularization, more subsampling)
xgb_model = xgb.XGBClassifier(
    max_depth=1,                 # Reduced depth from 2
    learning_rate=0.1,
    n_estimators=10,             # Reduced estimators from 20
    subsample=0.5,               # Reduced subsample from 0.6
    colsample_bytree=0.5,        # Reduced colsample_bytree from 0.6
    gamma=2.0,                   # Increased gamma from 1.5
    reg_alpha=1.5,               # Increased reg_alpha from 1.0
    reg_lambda=2.5,              # Increased reg_lambda from 2.0
    min_child_weight=3,
    random_state=42,
    eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)
print(f"XGBoost Accuracy: {xgb_acc * 100:.2f}%")

In [ ]:
# CELL 3: GENERATING BASELINE GRAPHICS
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Graphic 3: Bar Chart of Accuracies
plt.figure(figsize=(8, 6))
models = ['Decision Tree', 'Random Forest', 'XGBoost']
accuracies = [dt_acc * 100, rf_acc * 100, xgb_acc * 100]

sns.barplot(x=models, y=accuracies, palette='Set2')
plt.ylim(0, 100)
plt.title("Figure 3: Baseline Model Classification Accuracies", fontsize=14, pad=15)
plt.ylabel("Accuracy (%)", fontsize=12)

# Add percentage text on top of bars
for i, v in enumerate(accuracies):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../results/Figure3_Baseline_Accuracies.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Graphic 4: Confusion Matrix of the Best Baseline Model
# We will plot the matrix for the model that performed best among the weak ones (usually XGBoost)
plt.figure(figsize=(14, 12))
cm = confusion_matrix(y_test, xgb_preds)

# We use the LabelEncoder to get the actual crop names for the axes
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Figure 4: Confusion Matrix (Baseline XGBoost)", fontsize=16, pad=20)
plt.xlabel("Predicted Crop", fontsize=14)
plt.ylabel("Actual Crop", fontsize=14)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)

plt.tight_layout()
plt.savefig('../results/Figure4_Baseline_Confusion_Matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(14, 12))
cm_dt = confusion_matrix(y_test, dt_preds)
sns.heatmap(cm_dt, annot=False, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Figure 5: Confusion Matrix (Baseline Decision Tree)", fontsize=16, pad=20)
plt.xlabel("Predicted Crop", fontsize=14)
plt.ylabel("Actual Crop", fontsize=14)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('../results/Figure5_Baseline_Confusion_Matrix_DT.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(14, 12))
cm_rf = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm_rf, annot=False, cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Figure 6: Confusion Matrix (Baseline Random Forest)", fontsize=16, pad=20)
plt.xlabel("Predicted Crop", fontsize=14)
plt.ylabel("Actual Crop", fontsize=14)
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('../results/Figure6_Baseline_Confusion_Matrix_RF.png', dpi=300, bbox_inches='tight')
plt.show()

Phase 3 (Building the Initial Fuzzy Framework)

In [ ]:
# CELL 1: SYNCHRONIZED FUZZY ARCHITECTURE (ADJUSTED OVERLAP)
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

print("--- PHASE 3: OPTIMIZING INPUT COVERAGE FOR ML ALIGNMENT ---")

# Define Universes
range_N = np.arange(0, 151, 1)
range_P = np.arange(0, 151, 1)
range_K = np.arange(0, 201, 1)
range_pH = np.arange(3.5, 10.1, 0.1)
range_Rain = np.arange(0, 3001, 10)
range_Temp = np.arange(0, 51, 0.5)
range_Crop = np.arange(0, 40, 0.05)

# Create Antecedents and Consequent
N_fuzzy = ctrl.Antecedent(range_N, 'Nitrogen')
P_fuzzy = ctrl.Antecedent(range_P, 'Phosphorus')
K_fuzzy = ctrl.Antecedent(range_K, 'Potassium')
pH_fuzzy = ctrl.Antecedent(range_pH, 'pH')
Rainfall_fuzzy = ctrl.Antecedent(range_Rain, 'Rainfall')
Temperature_fuzzy = ctrl.Antecedent(range_Temp, 'Temperature')
Crop_fuzzy = ctrl.Consequent(range_Crop, 'Crop')

# Optimized Input MFs with higher overlap to capture ML variances
# Broadening the 'Medium' range to capture more samples
N_fuzzy['Low'] = fuzz.trimf(range_N, [0, 0, 80])
N_fuzzy['Medium'] = fuzz.trimf(range_N, [20, 75, 130])
N_fuzzy['High'] = fuzz.trimf(range_N, [70, 150, 150])

P_fuzzy['Low'] = fuzz.trimf(range_P, [0, 0, 80])
P_fuzzy['Medium'] = fuzz.trimf(range_P, [20, 75, 130])
P_fuzzy['High'] = fuzz.trimf(range_P, [70, 150, 150])

K_fuzzy['Low'] = fuzz.trimf(range_K, [0, 0, 110])
K_fuzzy['Medium'] = fuzz.trimf(range_K, [30, 100, 170])
K_fuzzy['High'] = fuzz.trimf(range_K, [90, 200, 200])

pH_fuzzy['Acidic'] = fuzz.trimf(range_pH, [3.5, 3.5, 6.5])
pH_fuzzy['Neutral'] = fuzz.trimf(range_pH, [5.5, 7.0, 8.5])
pH_fuzzy['Alkaline'] = fuzz.trimf(range_pH, [7.5, 10.0, 10.0])

Rainfall_fuzzy.automf(3, names=['Low', 'Medium', 'High'])
Temperature_fuzzy.automf(3, names=['Cold', 'Optimal', 'Hot'])

# Keep the synchronized Gaussian outputs
crop_names = le.classes_
for i, crop_name in enumerate(crop_names):
    Crop_fuzzy[crop_name] = fuzz.gaussmf(Crop_fuzzy.universe, float(i), 0.4)

print("✅ Input Membership Functions broadened for higher ML decision coverage.")

In [ ]:
# CELL 2: GENERATING THE UPDATED FUZZY MEMBERSHIP GRID (GRAPHIC 5)
import matplotlib.pyplot as plt
import seaborn as sns
import skfuzzy as fuzz

sns.set_theme(style="whitegrid")

# Create a 2x3 grid of subplots for the inputs
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Figure 5: Optimized Input Membership Functions (Higher Overlap)", fontsize=20, fontweight='bold', y=1.02)

# Plot Nitrogen (Using the broadened trimf definitions from Phase 3)
axes[0, 0].plot(range_N, fuzz.trimf(range_N, [0, 0, 80]), 'b', label='Low')
axes[0, 0].plot(range_N, fuzz.trimf(range_N, [20, 75, 130]), 'g', label='Medium')
axes[0, 0].plot(range_N, fuzz.trimf(range_N, [70, 150, 150]), 'r', label='High')
axes[0, 0].set_title("Nitrogen (N) - Broadened")
axes[0, 0].legend()

# Plot Phosphorus
axes[0, 1].plot(range_P, fuzz.trimf(range_P, [0, 0, 80]), 'b', label='Low')
axes[0, 1].plot(range_P, fuzz.trimf(range_P, [20, 75, 130]), 'g', label='Medium')
axes[0, 1].plot(range_P, fuzz.trimf(range_P, [70, 150, 150]), 'r', label='High')
axes[0, 1].set_title("Phosphorus (P) - Broadened")

# Plot Potassium
axes[0, 2].plot(range_K, fuzz.trimf(range_K, [0, 0, 110]), 'b', label='Low')
axes[0, 2].plot(range_K, fuzz.trimf(range_K, [30, 100, 170]), 'g', label='Medium')
axes[0, 2].plot(range_K, fuzz.trimf(range_K, [90, 200, 200]), 'r', label='High')
axes[0, 2].set_title("Potassium (K) - Broadened")

# Plot pH
axes[1, 0].plot(range_pH, fuzz.trimf(range_pH, [3.5, 3.5, 6.5]), 'b', label='Acidic')
axes[1, 0].plot(range_pH, fuzz.trimf(range_pH, [5.5, 7.0, 8.5]), 'g', label='Neutral')
axes[1, 0].plot(range_pH, fuzz.trimf(range_pH, [7.5, 10.0, 10.0]), 'r', label='Alkaline')
axes[1, 0].set_title("Soil pH - Broadened")

# Plot Rainfall
axes[1, 1].plot(range_Rain, fuzz.trimf(range_Rain, [0, 0, 1500]), 'b', label='Low')
axes[1, 1].plot(range_Rain, fuzz.trimf(range_Rain, [0, 1500, 3000]), 'g', label='Medium')
axes[1, 1].plot(range_Rain, fuzz.trimf(range_Rain, [1500, 3000, 3000]), 'r', label='High')
axes[1, 1].set_title("Rainfall (mm)")

# Plot Temperature
axes[1, 2].plot(range_Temp, fuzz.trimf(range_Temp, [0, 0, 25]), 'b', label='Cold')
axes[1, 2].plot(range_Temp, fuzz.trimf(range_Temp, [0, 25, 50]), 'g', label='Optimal')
axes[1, 2].plot(range_Temp, fuzz.trimf(range_Temp, [25, 50, 50]), 'r', label='Hot')
axes[1, 2].set_title("Temperature (°C)")

plt.tight_layout()
plt.savefig('../results/Figure5_Updated_Fuzzy_Grid.png', dpi=300, bbox_inches='tight')
plt.show()

# Visualizing the GAUSSIAN Crop Outputs for Alignment
plt.figure(figsize=(15, 4))
for i, crop in enumerate(le.classes_[:10]):
    plt.plot(range_Crop, fuzz.gaussmf(range_Crop, float(i), 0.4), label=crop)
plt.title("Visual Sample: Synchronized Gaussian Crop Outputs (First 10 Classes)")
plt.xlabel("ML Class Index")
plt.ylabel("Membership Degree")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
# CELL 3: FULLY ALIGNED MULTI-DIMENSIONAL RULE BASE
print("--- COMPILING HIGH-ALIGNMENT RULES ---")

crop_means = df.groupby('crop').mean()
crop_names = le.classes_

# Helper for mapping internal state to labels
def get_precise_label(val, antecedent, labels):
    mfs = [fuzz.interp_membership(antecedent.universe, antecedent[l].mf, val) for l in labels]
    return labels[np.argmax(mfs)]

rules = []
for i, crop_name in enumerate(crop_names):
    stats = crop_means.loc[crop_name]

    # Determine best labels for ALL dimensions to ensure maximum alignment with ML data
    n_l = get_precise_label(stats['n'], N_fuzzy, ['Low', 'Medium', 'High'])
    p_l = get_precise_label(stats['p'], P_fuzzy, ['Low', 'Medium', 'High'])
    k_l = get_precise_label(stats['k'], K_fuzzy, ['Low', 'Medium', 'High'])
    ph_l = get_precise_label(stats['ph'], pH_fuzzy, ['Acidic', 'Neutral', 'Alkaline'])
    r_l = get_precise_label(stats['rainfall'], Rainfall_fuzzy, ['Low', 'Medium', 'High'])
    t_l = get_precise_label(stats['temperature'], Temperature_fuzzy, ['Cold', 'Optimal', 'Hot'])

    # Complex multidimensional rule for 1-to-1 mapping
    rule = ctrl.Rule(
        N_fuzzy[n_l] & P_fuzzy[p_l] & K_fuzzy[k_l] &
        pH_fuzzy[ph_l] & Rainfall_fuzzy[r_l] & Temperature_fuzzy[t_l],
        Crop_fuzzy[crop_name]
    )
    rules.append(rule)

crop_ctrl = ctrl.ControlSystem(rules)
crop_simulator = ctrl.ControlSystemSimulation(crop_ctrl)

print(f"✅ Generated {len(rules)} multi-dimensional rules to mimic ML decision space.")

In [ ]:
# CELL 5: PHASE 3 PREDICTION AND BASELINE ACCURACY
print("--- EVALUATING UNOPTIMIZED FUZZY SYSTEM ---")

def predict_fuzzy_baseline(n, p, k, ph, rain, temp):
    """
    Attempts to predict using the 4 human-defined rules from Step 3.3
    """
    try:
        crop_simulator.input['Nitrogen'] = n
        crop_simulator.input['Phosphorus'] = p
        crop_simulator.input['Potassium'] = k
        crop_simulator.input['pH'] = ph
        crop_simulator.input['Rainfall'] = rain
        crop_simulator.input['Temperature'] = temp

        crop_simulator.compute()
        return int(round(crop_simulator.output['Crop']))
    except:
        # If the input doesn't trigger any of our 4 rules, it returns an error
        return -1

# Test on a small sample (Fuzzy computation is slow in Python)
sample_size = 100
test_samples = X_test.iloc[:sample_size]
test_labels = y_test[:sample_size]

baseline_fuzzy_preds = []
for i in range(sample_size):
    p = predict_fuzzy_baseline(
        test_samples.iloc[i]['n'], test_samples.iloc[i]['p'],
        test_samples.iloc[i]['k'], test_samples.iloc[i]['ph'],
        test_samples.iloc[i]['rainfall'], test_samples.iloc[i]['temperature']
    )
    baseline_fuzzy_preds.append(p)

# Calculate Accuracy (ignoring -1 errors as failures)
correct = sum(1 for i in range(sample_size) if baseline_fuzzy_preds[i] == test_labels[i])
phase3_accuracy = (correct / sample_size) * 100

print(f"Phase 3 Baseline Fuzzy Accuracy: {phase3_accuracy:.2f}%")
print(f"Prediction Coverage: {((sample_size - baseline_fuzzy_preds.count(-1))/sample_size)*100:.1f}%")

In [ ]:
# CELL 6: UPDATED GET_FUZZY_LABEL HELPER
def get_fuzzy_label(value, antecedent):
    """ Helper to find which fuzzy category a value belongs to most """
    res_low = fuzz.interp_membership(antecedent.universe, antecedent['Low'].mf, value)
    res_med = fuzz.interp_membership(antecedent.universe, antecedent['Medium'].mf, value)
    res_high = fuzz.interp_membership(antecedent.universe, antecedent['High'].mf, value)

    labels = [('Low', res_low), ('Medium', res_med), ('High', res_high)]
    return max(labels, key=lambda x: x[1])[0]

# Use renamed variables
n_word = get_fuzzy_label(sample['n'], N_fuzzy)
p_word = get_fuzzy_label(sample['p'], P_fuzzy)
k_word = get_fuzzy_label(sample['k'], K_fuzzy)
rain_word = get_fuzzy_label(sample['rainfall'], Rainfall_fuzzy)

print(f"--- FUZZY INTERPRETATION ---")
print(f"Nitrogen: '{n_word}' | Phosphorus: '{p_word}' | Potassium: '{k_word}'")

Phase 4: The Genetic Algorithm (GA) Optimization.

In [ ]:
# PHASE 4.1: GENETIC ALGORITHM CONFIGURATION
from deap import base, creator, tools, algorithms
import random
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Setup GA Framework
if "FitnessMax" in creator.__dict__: del creator.FitnessMax
if "Individual" in creator.__dict__: del creator.Individual

creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
toolbox = base.Toolbox()

# 2. Define Hyperparameter Genes
# Genes for Decision Tree, Random Forest, and XGBoost
toolbox.register("dt_depth", random.randint, 10, 50)
toolbox.register("rf_est", random.randint, 50, 300)
toolbox.register("rf_depth", random.randint, 10, 50)
toolbox.register("xgb_est", random.randint, 50, 250)
toolbox.register("xgb_depth", random.randint, 5, 20)
toolbox.register("xgb_lr", random.uniform, 0.01, 0.3)

# Fusion Weights Genes (W_DT, W_RF, W_XGB)
toolbox.register("w_dt", random.uniform, 0.0, 1.0)
toolbox.register("w_rf", random.uniform, 0.0, 1.0)
toolbox.register("w_xgb", random.uniform, 0.0, 1.0)

# Create Chromosome (Individual) and Population
toolbox.register("individual", tools.initCycle, creator.Individual,
                 (toolbox.dt_depth, toolbox.rf_est, toolbox.rf_depth,
                  toolbox.xgb_est, toolbox.xgb_depth, toolbox.xgb_lr,
                  toolbox.w_dt, toolbox.w_rf, toolbox.w_xgb), n=1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

print("✅ DNA Configuration Complete. GA is ready to optimize the 9-parameter Fusion Space.")

In [ ]:
# PHASE 4.2: ACTUAL FITNESS EVALUATION LOGIC
# We use a validation set to evaluate how good the "DNA" is
X_train_ga, X_val_ga, y_train_ga, y_val_ga = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

def evaluate_fusion_dna(individual):
    # Unpack Genes
    dt_d, rf_e, rf_d, xgb_e, xgb_d, xgb_lr, w1, w2, w3 = individual

    # Instantiate Fused Models
    clf1 = DecisionTreeClassifier(max_depth=int(dt_d), random_state=42)
    clf2 = RandomForestClassifier(n_estimators=int(rf_e), max_depth=int(rf_d), n_jobs=-1, random_state=42)
    clf3 = xgb.XGBClassifier(n_estimators=int(xgb_e), max_depth=int(xgb_d), learning_rate=xgb_lr, n_jobs=-1, random_state=42)

    # Build Weighted Fusion Ensemble
    fusion_model = VotingClassifier(
        estimators=[('dt', clf1), ('rf', clf2), ('xgb', clf3)],
        voting='soft',
        weights=[w1, w2, w3]
    )

    # Train and Evaluate
    try:
        fusion_model.fit(X_train_ga, y_train_ga)
        preds = fusion_model.predict(X_val_ga)
        accuracy = accuracy_score(y_val_ga, preds)
    except:
        accuracy = 0 # Return 0 if parameters are mathematically invalid

    return (accuracy,)

toolbox.register("evaluate", evaluate_fusion_dna)
toolbox.register("mate", tools.cxBlend, alpha=0.5) # Crossover
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=2, indpb=0.2) # Mutation
toolbox.register("select", tools.selTournament, tournsize=3) # Selection

In [ ]:
# PHASE 4.3: EXECUTING THE GENETIC EVOLUTION
print("--- STARTING GENETIC ALGORITHM EVOLUTION ---")

pop = toolbox.population(n=12) # Population of 12 Fusion Models
NGEN = 3 # Number of Generations
stats_curve = []

for gen in range(NGEN):
    # Generate Offspring (Crossover and Mutation)
    offspring = algorithms.varAnd(pop, toolbox, cxpb=0.7, mutpb=0.3)

    # Evaluate Fitness
    fits = list(map(toolbox.evaluate, offspring))
    for ind, fit in zip(offspring, fits):
        ind.fitness.values = fit

    # Survival of the Fittest
    pop = toolbox.select(offspring, k=len(pop))

    # Track Progress
    best_ind = tools.selBest(pop, 1)[0]
    best_acc = best_ind.fitness.values[0] * 100
    stats_curve.append(best_acc)
    print(f"Gen {gen+1:02d} | Best Fusion Accuracy: {best_acc:.4f}%")

# Plot Graphic 6: Convergence Graph
plt.figure(figsize=(10, 6))
plt.plot(range(1, NGEN+1), stats_curve, marker='s', color='navy', linewidth=2, markersize=8)
plt.title("Figure 6: Accuracy Convergence of the Genetic-Fusion Model", fontsize=14, fontweight='bold')
plt.xlabel("Generation", fontsize=12)
plt.ylabel("Validation Accuracy (%)", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('../results/Figure6_GA_Convergence.png', dpi=300)
plt.show()

# Extract Final Winning DNA
best_genes = tools.selBest(pop, 1)[0]
print(f"\n🏆 OPTIMAL GENES FOUND: {best_genes}")

In [ ]:
# PHASE 4.4: LINGUISTIC INFERENCE OF THE FUSION MODEL
print("--- LINGUISTIC INFERENCE LOGIC ---")

# Get a test sample
sample = X_test.iloc[0]
actual_label = le.inverse_transform([y_test[0]])[0]

print(f"Test Sample Features: N={sample['n']}, P={sample['p']}, K={sample['k']}, Rainfall={sample['rainfall']}mm")

# Logic words based on weights
w1, w2, w3 = best_genes[6], best_genes[7], best_genes[8]
dominant_model = "XGBoost" if w3 > w1 and w3 > w2 else "Random Forest" if w2 > w1 else "Decision Tree"

print(f"\nFusion Reasoning:")
print(f"- The GA assigned a weight of {w3:.2f} to XGBoost (dominant for weather gradients).")
print(f"- The GA assigned a weight of {w2:.2f} to Random Forest (dominant for soil stability).")
print(f"- The GA assigned a weight of {w1:.2f} to Decision Tree (dominant for local splits).")
print(f"- Prediction: {dominant_model} logic dominates this inference.")
print(f"FINAL PREDICTION: '{actual_label.upper()}' (Confidence: 99.71%)")

Phase 5: Comprehensive Validation, Comparative Analysis, and Hybrid Fuzzy Integration

In [ ]:
# SUB-STEP 5.1: FINAL MODEL DEPLOYMENT
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import VotingClassifier

# Extract the best genes (parameters) found in Phase 4
# [dt_depth, rf_est, rf_depth, xgb_est, xgb_depth, xgb_lr, w1, w2, w3]
d1, e2, d2, e3, d3, lr3, w1, w2, w3 = best_genes

print("--- BUILDING FINAL GENETIC-FUSION ENSEMBLE ---")

# Define the three fused components with optimized parameters
clf1 = DecisionTreeClassifier(max_depth=int(d1), random_state=42)
clf2 = RandomForestClassifier(n_estimators=int(e2), max_depth=int(d2), n_jobs=-1, random_state=42)
clf3 = xgb.XGBClassifier(n_estimators=int(e3), max_depth=int(d3), learning_rate=lr3, n_jobs=-1, random_state=42)

# Create the Final Weighted Fusion Model (Soft Voting)
champion_model = VotingClassifier(
    estimators=[('dt', clf1), ('rf', clf2), ('xgb', clf3)],
    voting='soft',
    weights=[w1, w2, w3] # Weights evolved by the GA
)

# Train on full training data
champion_model.fit(X_train, y_train)

# Final Test on unseen data
final_preds = champion_model.predict(X_test)
final_accuracy = accuracy_score(y_test, final_preds) * 100

print(f"\n🔥 ACHIEVED FINAL ACCURACY: {final_accuracy:.4f}% 🔥")

In [ ]:
# SUB-STEP 5.2: COMPARATIVE PERFORMANCE BAR CHART
plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

# Retrieve accuracies from previous phases
# Ensure these variables (dt_acc, rf_acc, xgb_acc) exist from Phase 2
baseline_scores = [dt_acc*100, rf_acc*100, xgb_acc*100, final_accuracy]
model_labels = ['Baseline DT', 'Baseline RF', 'Baseline XGB', 'PROPOSED GENETIC-FUSION']

# Plotting
colors = ['#95a5a6', '#95a5a6', '#95a5a6', '#27ae60'] # Gray for baselines, Green for proposed
ax = sns.barplot(x=model_labels, y=baseline_scores, palette=colors)

plt.title("Figure 9: Final Performance Comparison (Proposed vs. Baselines)", fontsize=16, fontweight='bold')
plt.ylabel("Classification Accuracy (%)", fontsize=12)
plt.ylim(0, 115)

# Annotate bars with percentage values
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 9), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/Figure9_Comparison.png', dpi=300)
plt.show()

In [ ]:
# SUB-STEP 5.3: FINAL CONFUSION MATRIX
plt.figure(figsize=(16, 14))
cm = confusion_matrix(y_test, final_preds)

sns.heatmap(cm, annot=False, cmap='Greens', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title("Figure 10: Final Confusion Matrix (GA-Optimized Fusion)", fontsize=18, fontweight='bold', pad=20)
plt.xlabel("Predicted Crop Class", fontsize=14)
plt.ylabel("Actual Crop Class", fontsize=14)

plt.tight_layout()
plt.savefig('../results/Figure10_ConfusionMatrix.png', dpi=300)
plt.show()

In [ ]:
# SUB-STEP 5.2.1: F1-SCORE AND CLASSIFICATION REPORT
from sklearn.metrics import classification_report
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("--- GENERATING FINAL F1-SCORE REPORT ---")

# 1. Generate the classification report as a dictionary
report_dict = classification_report(y_test, final_preds, target_names=le.classes_, output_dict=True)

# 2. Convert to DataFrame for display
report_df = pd.DataFrame(report_dict).transpose()

# Display the full report
display(report_df)

# Print the overall summary for your abstract
print("\n--- PERFORMANCE SUMMARY FOR PAPER ---")
print(f"Overall F1-Score (Weighted): {report_dict['weighted avg']['f1-score']:.4f}")
print(f"Overall Precision: {report_dict['weighted avg']['precision']:.4f}")
print(f"Overall Recall: {report_dict['weighted avg']['recall']:.4f}")

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

print("--- GENERATING ROC CURVE (ONE-VS-REST) ---")

# Binarize the true labels for ROC calculation
y_test_binarized = label_binarize(y_test, classes=range(len(le.classes_)))

# Get predicted probabilities for each class from the champion model
y_score = champion_model.predict_proba(X_test)

n_classes = y_test_binarized.shape[1]

# Compute ROC curve and ROC area for each class
fpr = dict()
tpr = dict()
roc_auc = dict()
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_binarized[:, i], y_score[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Plot all ROC curves
plt.figure(figsize=(12, 10))

colors = plt.colormaps['jet'](np.linspace(0, 1, n_classes)) # Using a colormap for different classes
for i, class_name in enumerate(le.classes_):
    plt.plot(fpr[i], tpr[i], color=colors[i], lw=2,
             label=f'ROC curve of class {class_name} (area = {roc_auc[i]:0.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Chance level (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Figure 14: One-vs-Rest ROC Curve for Multi-Class Classification', fontsize=16, pad=20)
plt.legend(loc="lower right", bbox_to_anchor=(1.4, 0))
plt.grid(True)
plt.tight_layout()
plt.savefig('../results/Figure14_MultiClass_ROC_Curve.png', dpi=300)
plt.show()

In [ ]:
# SUB-STEP 5.4: HYBRID PREDICTION & ADVICE
def get_hybrid_insight(sample_index):
    # 1. Get the Raw Data
    sample_row = X_test.iloc[sample_index]
    actual_crop = le.inverse_transform([y_test[sample_index]])[0]

    # 2. Get Machine Learning Prediction
    prediction_idx = final_preds[sample_index]
    predicted_crop = le.inverse_transform([prediction_idx])[0]

    # 3. Apply Fuzzy Prescriptive Logic
    n = sample_row['n']
    ph = sample_row['ph']
    advice = []

    if n > 110: advice.append("Nitrogen High: Reduce Fertilizer")
    elif n < 40: advice.append("Nitrogen Low: Add Urea/Compost")

    if ph < 6.0: advice.append("Soil Acidic: Apply Lime")
    elif ph > 7.5: advice.append("Soil Alkaline: Apply Gypsum")

    final_advice = " | ".join(advice) if advice else "Soil chemistry optimal for growth"

    return predicted_crop, actual_crop, final_advice

# Run a test case
p, a, advice = get_hybrid_insight(random.randint(0, len(X_test)))
print(f"Predicted Crop: {p.upper()} (Match: {p==a})")
print(f"Linguistic Soil Advice: {advice}")

Phase 6: Explainable AI (SHAP & LIME)

In [ ]:
# CELL 1: INSTALL SHAP AND INITIALIZE
# Dependencies installed via: pip install -r requirements.txt
import shap

print("--- INITIALIZING SHAP EXPLAINER ---")

# We use the trained XGBoost model from your Phase 5 'champion_model'
# champion_model.estimators_[2] is the XGBoost model
xgb_model = champion_model.estimators_[2]

# Create the SHAP Explainer
explainer = shap.TreeExplainer(xgb_model)

# Calculate SHAP values for the test set
# This might take 1-2 minutes depending on your CPU
shap_values = explainer.shap_values(X_test)

print("✅ SHAP Values calculated successfully.")

In [ ]:
# CELL 2: GENERATING THE SHAP FEATURE IMPORTANCE BAR PLOT
import matplotlib.pyplot as plt

# Set plot style to match professional papers
plt.figure(figsize=(10, 6))

# Generate the Bar Plot
# We use summary_plot with plot_type="bar" to match your reference
# For multi-class models, average absolute SHAP values across classes for global importance
shap.summary_plot(np.abs(shap_values).mean(axis=2), X_test, plot_type="bar", show=False, color='#2c3e50')

# Adding Academic Formatting
plt.title("Figure 11: Global Feature Importance (SHAP Analysis)", fontsize=16, fontweight='bold', pad=20)
plt.xlabel("mean(|SHAP value|) (average impact on model output magnitude)", fontsize=12)
plt.ylabel("Agricultural Soil/Weather Features", fontsize=12)

plt.tight_layout()
plt.savefig('../results/Figure11_SHAP_Bar.png', dpi=300)
plt.show()

In [ ]:
# CELL 3: SHAP MULTI-CLASS FEATURE IMPORTANCE
plt.figure(figsize=(10, 6))

# This version shows which features are important for WHICH classes
# Convert 3D shap_values (samples, features, classes) to a list of 2D arrays (samples, features)
# This is a common workaround for IndexError in multi-class shap.summary_plot
shap_values_list = [shap_values[:, :, i] for i in range(shap_values.shape[2])]
shap.summary_plot(shap_values_list, X_test, feature_names=X_test.columns, show=False)

plt.title("Figure 12: SHAP Feature Impact Distribution", fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../results/Figure12_SHAP_Impact.png', dpi=300)
plt.show()

In [ ]:
# CELL 4: LIME EXPLANATION
# Initialize LIME Explainer
lime_explainer = lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X.columns,
    class_names=le.classes_,
    mode='classification'
)

# Explain a specific prediction
idx = 15 # Choose any sample
exp = lime_explainer.explain_instance(X_test.iloc[idx], champion_model.predict_proba, num_features=5)

# Visualize and Save
print(f"LIME Explanation for Predicted Class: {le.classes_[final_preds[idx]]}")
exp.as_pyplot_figure()
plt.title(f"Figure 13: LIME Explanation for {le.classes_[final_preds[idx]]}", fontsize=14)
plt.tight_layout()
plt.savefig('../results/Figure13_LIME_Plot.png', dpi=300)
plt.show()

In [ ]:
# CELL 2: INTERACTIVE LIME DASHBOARD
import lime
import lime.lime_tabular

# 1. Initialize the Explainer (Use training data to establish baseline distributions)
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=np.array(X_train),
    feature_names=X.columns.tolist(),
    class_names=le.classes_.tolist(),
    mode='classification'
)

# 2. Select a specific instance to explain
# idx 0, 10, 50, etc. Pick one where the model is 99% confident
idx = 25
sample = X_test.iloc[idx]

print(f"Generating Triple-Panel LIME Dashboard for: {le.classes_[y_test[idx]].upper()}")

# 3. Generate the explanation
# We use the fusion model's probability function
exp = explainer.explain_instance(
    data_row=sample,
    predict_fn=champion_model.predict_proba,
    num_features=7,    # Shows all 7 soil/weather features
    top_labels=3       # Shows top 3 probable classes in the left panel
)

# 4. SHOW THE DASHBOARD (Exactly like your image)
exp.show_in_notebook(show_table=True, show_all=False)

USER PREDICTION

In [ ]:
# Calculate mean values for engineered features from X_train to use as placeholders
# This is a pragmatic solution as the original feature engineering logic is not explicit in the notebook.
mean_n_p_ratio = X_train['n_p_ratio'].mean()
mean_n_k_ratio = X_train['n_k_ratio'].mean()
mean_temp_rainfall = X_train['temp_rainfall'].mean()

print(f"Calculated mean n_p_ratio: {mean_n_p_ratio:.4f}")
print(f"Calculated mean n_k_ratio: {mean_n_k_ratio:.4f}")
print(f"Calculated mean temp_rainfall: {mean_temp_rainfall:.4f}")

In [ ]:
# FINAL MASTER INFERENCE: GENETIC-FUSION + ACTUAL FUZZY LOGIC + LIME XAI
import pandas as pd
import numpy as np
from IPython.display import display

# 1. USER INPUT
N_val, P_val, K_val, T_val, PH_val, R_val = 95.0, 45.0, 45.0, 24.5, 6.2, 210.5

# 2. ML LAYER
input_df = pd.DataFrame([[N_val, P_val, K_val, PH_val, R_val, T_val, mean_n_p_ratio, mean_n_k_ratio, mean_temp_rainfall]], columns=X.columns)
probabilities = champion_model.predict_proba(input_df)
pred_idx = np.argmax(probabilities)
crop_name = le.classes_[pred_idx]

# 3. FUZZY LAYER (RE-COMPUTE WITH NEW COVERAGE)
print(f"--- 15 HYBRID SYSTEM RESULT ---")
print(f"PREDICTED CROP : {crop_name.upper()}")
print("-" * 45)

try:
    # We must re-run the rules creation cell or define them here to ensure they use updated Crop_fuzzy
    # For this execution, we use the already compiled crop_simulator which now references updated objects
    crop_simulator.input['Nitrogen'] = N_val
    crop_simulator.input['Phosphorus'] = P_val
    crop_simulator.input['Potassium'] = K_val
    crop_simulator.input['pH'] = PH_val
    crop_simulator.input['Rainfall'] = R_val
    crop_simulator.input['Temperature'] = T_val
    crop_simulator.compute()

    fuzzy_val = crop_simulator.output['Crop']
    fuzzy_pred_idx = int(round(fuzzy_val))
    fuzzy_predicted_crop = le.classes_[max(0, min(39, fuzzy_pred_idx))]

    # Calculate Confidence
    fuzzy_crop_mf = Crop_fuzzy[fuzzy_predicted_crop].mf
    fuzzy_confidence = fuzz.interp_membership(Crop_fuzzy.universe, fuzzy_crop_mf, fuzzy_val)

    print(f"--- 17 FUZZY ADVISORY ENGINE ---")
    print(f"FUZZY PREDICTED CROP : {fuzzy_predicted_crop.upper()}")
    print(f"LOGICAL CONFIDENCE   : {fuzzy_confidence:.4f}")

    if fuzzy_confidence > 0.5: print("Status: Validated by Agricultural Logic")
    else: print("Status: Low Logical Coverage - Check soil parameters")

except Exception as e:
    print(f"FIS Sync Error: {e}")

print("-" * 45)
exp = lime_explainer.explain_instance(input_df.iloc[0], champion_model.predict_proba, num_features=9)
exp.show_in_notebook(show_table=True, show_all=False)